# PolyAletheia - Training on Colab

Run this notebook on Google Colab to train using GPU.
**NOTE:** Please upload `train_split.csv` and `val_split.csv` when prompted.

In [ ]:
# 1. Install dependencies
!pip install torch transformers rdkit pandas numpy wandb

import os
try:
    from google.colab import files
    if not os.path.exists("train_split.csv") or not os.path.exists("val_split.csv"):
        print("Upload train_split.csv and val_split.csv...")
        files.upload()
except ImportError:
    pass

In [ ]:
# 2. Imports & Setup
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_linear_schedule_with_warmup
from rdkit import Chem
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import wandb

# Check for GPU
if torch.cuda.is_available():
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print("WARNING: Running on CPU. Enable GPU in Runtime > Change runtime type.")
    device = torch.device("cpu")

In [ ]:
# 3. SMILES Utilities (from smiles_func.py)

def is_valid_smiles(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return mol is not None

def canonicalize(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    return Chem.MolToSmiles(mol)


In [ ]:
# 4. Tokenizer & Model (from tokenizer.py and model.py)

MODEL_NAME = "seyonec/ChemBERTa-zinc-base-v1"

def get_tokenizer():
    return AutoTokenizer.from_pretrained(MODEL_NAME)

class PolymerPredictor(nn.Module):
    def __init__(self, model_name=MODEL_NAME, num_tasks=5):
        super().__init__()
        print(f"Loading {model_name}...")
        self.backbone = AutoModel.from_pretrained(model_name)
        
        hidden_size = self.backbone.config.hidden_size
        
        # simple regression head
        self.head = nn.Sequential(
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, num_tasks)
        )
        
    def forward(self, input_ids, attention_mask=None, **kwargs):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        predictions = self.head(cls_embedding)
        return predictions

In [ ]:
# 5. Dataset & Loss (from train.py)

PROPS = ["Tg", "FFV", "Tc", "Density", "Rg"]

class PolymerDataset(Dataset):
    def __init__(self, csv_path, tokenizer, max_len=128):
        try:
            self.df = pd.read_csv(csv_path)
        except FileNotFoundError:
            print(f"ERROR: {csv_path} not found. Please upload it to Colab (drag & drop to files sidebar).")
            raise
            
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.targets = self.df[PROPS].values.astype(np.float32)
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        smiles = row["SMILES"]
        
        encoding = self.tokenizer(
            smiles,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        return {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten(),
            "targets": torch.tensor(self.targets[idx])
        }

def masked_mse_loss(preds, targets):
    mask = ~torch.isnan(targets)
    diff = preds[mask] - targets[mask]
    loss = (diff ** 2).mean()
    if torch.isnan(loss):
        return torch.tensor(0.0, requires_grad=True).to(preds.device)
    return loss

In [ ]:
# 6. Training Loop Helpers

def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    progress = tqdm(loader, desc="Training")
    for batch in progress:
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        targets = batch["targets"].to(device)
        
        optimizer.zero_grad()
        preds = model(input_ids, mask)
        loss = masked_mse_loss(preds, targets)
        loss.backward()
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        progress.set_postfix({"loss": loss.item()})
        
        if wandb.run is not None:
            wandb.log({"train_loss": loss.item()})
            
    return total_loss / len(loader)

def validate_epoch(model, loader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            mask = batch["attention_mask"].to(device)
            targets = batch["targets"].to(device)
            preds = model(input_ids, mask)
            loss = masked_mse_loss(preds, targets)
            total_loss += loss.item()
    
    val_loss = total_loss / len(loader)
    if wandb.run is not None:
        wandb.log({"val_loss": val_loss})
    return val_loss

In [ ]:
# 7. Run Training

# Init WandB
wandb.login() # will prompt for key
wandb.init(project="polyaletheia-colab", config={
    "model": MODEL_NAME,
    "batch_size": 32,
    "epochs": 20
})

tokenizer = get_tokenizer()
model = PolymerPredictor().to(device)

print("Loading datasets...")
train_ds = PolymerDataset("train_split.csv", tokenizer)
val_ds = PolymerDataset("val_split.csv", tokenizer)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
val_dl = DataLoader(val_ds, batch_size=32)

optimizer = AdamW(model.parameters(), lr=1e-4)
epochs = 20
scheduler = get_linear_schedule_with_warmup(optimizer, 0, len(train_dl)*epochs)

best_val_loss = float("inf")

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    train_loss = train_epoch(model, train_dl, optimizer, scheduler, device)
    val_loss = validate_epoch(model, val_dl, device)
    print(f"Train Loss {train_loss:.4f}, Val Loss {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "best_model_colab.pth")
        print("Saved new best model.")
        wandb.log({"best_val_loss": best_val_loss})
        
wandb.finish()